# LLM Series — 1: Tokenisation
### How text becomes integers, and why the choice matters more than you'd think

---

## Where we are

This is the first notebook in the LLM series. Before a transformer can do anything — before attention, before the forward pass, before sampling — raw text has to be converted into a sequence of integers. That conversion is tokenisation, and it's the entry point into the entire pipeline.

The series builds like this:

```
Text → [THIS NOTEBOOK] → integers
     → Embeddings         → vectors
     → The Forward Pass   → probability distribution
     → Sampling           → generated text
     → Training at Scale  → how the weights got there
```

Understanding tokenisation properly explains a surprising number of things about how LLMs behave — why they struggle with certain arithmetic, why they sometimes make odd spelling errors, why some languages cost more tokens than others, and why prompt length isn't the same as character length.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import rcParams
from collections import Counter, defaultdict
import re

# ── Dark theme ────────────────────────────────────────────────────────────────
BG      = '#0d1117'
SURFACE = '#161b22'
ACCENT  = '#e05c5c'
TEXT    = '#c9d1d9'
MUTED   = '#484f58'
BLUE    = '#58a6ff'
GREEN   = '#3fb950'
YELLOW  = '#d29922'
PURPLE  = '#bc8cff'

rcParams.update({
    'figure.facecolor':  BG,
    'axes.facecolor':    SURFACE,
    'axes.edgecolor':    MUTED,
    'axes.labelcolor':   TEXT,
    'xtick.color':       TEXT,
    'ytick.color':       TEXT,
    'text.color':        TEXT,
    'grid.color':        MUTED,
    'grid.alpha':        0.3,
    'legend.facecolor':  SURFACE,
    'legend.edgecolor':  MUTED,
    'figure.dpi':        110,
})

np.random.seed(42)
print("Ready.")

## Part 1 — Why not just use characters or words?

The obvious approaches both have serious problems.

### Character-level tokenisation

Treat every character as a token. Vocabulary is small (~256 for ASCII, ~150k for full Unicode). But sequences become very long — "the quick brown fox" is 18 characters. Long sequences mean more attention computations (attention scales as $O(n^2)$ in sequence length — see the Attention notebook) and the model has to learn word-level meaning from scratch by composing characters.

### Word-level tokenisation

Split on whitespace and punctuation. Sequences are short and tokens carry semantic meaning. But:
- Vocabulary explodes: English has 170,000+ words, plus inflections, compounds, proper nouns, technical terms. A vocabulary this large makes the embedding matrix and unembedding matrix enormous.
- Out-of-vocabulary (OOV) problem: any word not seen in training gets mapped to `<UNK>`. The model can't handle it.
- "run", "runs", "running", "runner" are four different tokens with no shared representation, even though they're clearly related.

### Subword tokenisation: the practical solution

Decompose text into **subword units** — pieces that are larger than characters but smaller than words. Common words get their own token; rare words get split into recognisable pieces. "unhappiness" might become ["un", "happiness"] or ["un", "happy", "ness"].

Vocabulary size stays manageable (~32k–100k tokens). Sequences are shorter than character-level. The model can handle new words by composing known subwords. Related words share subword tokens and thus share some representation.

The dominant algorithm for learning subword vocabularies is **Byte Pair Encoding (BPE)**.

In [ ]:
# ── Compare tokenisation strategies on the same text ──────────────────────────
sample = "The researchers are running experiments on unhappiness and rerunning them."

# Character-level
char_tokens = list(sample)

# Word-level (simple split)
word_tokens = re.findall(r"[\w']+|[.,!?;]", sample)

# Approximate subword (we'll implement BPE properly below)
# For now, show the intuition
subword_approx = ['The', ' re', 'search', 'ers', ' are', ' run', 'ning',
                  ' experi', 'ments', ' on', ' un', 'happ', 'iness',
                  ' and', ' re', 'run', 'ning', ' them', '.']

print("Sample text:")
print(f'  "{sample}"')
print(f"  ({len(sample)} characters)\n")

print(f"Character-level: {len(char_tokens)} tokens")
print(f"  {char_tokens[:15]}...\n")

print(f"Word-level: {len(word_tokens)} tokens")
print(f"  {word_tokens}\n")

print(f"Subword (BPE-style): {len(subword_approx)} tokens")
print(f"  {subword_approx}")
print("\nNote: 'run', 'running', 'rerunning' all share the subword 'run'")
print("      'unhappiness' is decomposed into known pieces")

## Part 2 — Byte Pair Encoding: the algorithm

BPE was originally a data compression algorithm (Philip Gage, 1994). Sennrich et al. (2016) adapted it for neural machine translation, and it's now the basis of GPT-2, GPT-3, GPT-4, and many others.

The algorithm:

1. Start with a base vocabulary of individual characters (or bytes)
2. Count all adjacent pairs of symbols in the corpus
3. Merge the most frequent pair into a new symbol
4. Repeat steps 2–3 until the vocabulary reaches the desired size

The merges are **learned from the corpus**. Frequent character sequences get merged into single tokens early; rare sequences remain as character sequences.

This means the vocabulary reflects the actual distribution of the training text. English-heavy corpora produce tokens tuned for English. Code-heavy corpora produce tokens for common programming patterns. Common words like "the" and "and" become single tokens; obscure technical terms get split.

Let me implement this from scratch.

In [ ]:
# ── BPE from scratch ───────────────────────────────────────────────────────────

def get_vocab(corpus):
    """
    Convert a corpus into a frequency dictionary of character-split words.
    Words are represented as tuples of characters + end-of-word marker.
    The space prefix (Ġ) marks word boundaries.
    """
    vocab = defaultdict(int)
    for word in corpus.lower().split():
        # Represent each word as a sequence of characters
        chars = tuple(list(word) + ['</w>'])
        vocab[chars] += 1
    return dict(vocab)

def get_pairs(vocab):
    """Count all adjacent symbol pairs across all words."""
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        for i in range(len(word) - 1):
            pairs[(word[i], word[i+1])] += freq
    return dict(pairs)

def merge_vocab(pair, vocab):
    """Merge all instances of a symbol pair in the vocabulary."""
    merged = ''.join(pair)
    new_vocab = {}
    for word, freq in vocab.items():
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and word[i] == pair[0] and word[i+1] == pair[1]:
                new_word.append(merged)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_vocab[tuple(new_word)] = freq
    return new_vocab

def train_bpe(corpus, n_merges):
    """Train BPE on a corpus for n_merges iterations."""
    vocab  = get_vocab(corpus)
    merges = []

    for i in range(n_merges):
        pairs = get_pairs(vocab)
        if not pairs:
            break
        best_pair = max(pairs, key=pairs.get)
        vocab     = merge_vocab(best_pair, vocab)
        merges.append((best_pair, pairs[best_pair]))

    return vocab, merges


# Small corpus to make merges visible
corpus = """
the cat sat on the mat the cat ate the rat
the rat ran from the cat the mat was flat
a fat cat sat on a mat a rat ate a bat
the cat and the rat sat on the flat mat
low lower lowest new newer newest
running runner runs run
"""

vocab, merges = train_bpe(corpus, n_merges=20)

print("Top 20 BPE merges (most frequent pair → merged token):")
print(f"{'Merge #':>8s}  {'Pair':>20s}  {'Frequency':>10s}  {'Result':>12s}")
print("─" * 58)
for i, (pair, freq) in enumerate(merges[:20], 1):
    print(f"{i:>8d}  {str(pair):>20s}  {freq:>10d}  {''.join(pair):>12s}")

In [ ]:
# ── Visualise BPE learning step by step ───────────────────────────────────────
# Show how 'the' evolves from ['t','h','e','</w>'] to ['the</w>']

mini_corpus = "the cat sat on the mat the cat ate the rat the cat"
mini_vocab  = get_vocab(mini_corpus)

print("Initial vocabulary (character-level):")
for word, freq in sorted(mini_vocab.items(), key=lambda x: -x[1])[:8]:
    print(f"  {list(word)}  ×{freq}")

print("\nApplying merges step by step:")
v = mini_vocab.copy()
for step in range(8):
    pairs = get_pairs(v)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    v    = merge_vocab(best, v)
    print(f"  Step {step+1}: merge {best} → '{''.join(best)}'  (freq={pairs[best]})")

print("\nVocabulary after 8 merges:")
for word, freq in sorted(v.items(), key=lambda x: -x[1])[:8]:
    print(f"  {list(word)}  ×{freq}")

In [ ]:
# ── Visualise the merge tree ───────────────────────────────────────────────────
# Show how 'the' is assembled from characters through BPE merges

fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')
ax.set_xlim(0, 10)
ax.set_ylim(0, 4)

def token_box(ax, x, y, text, color=BLUE, width=1.2):
    bbox = dict(boxstyle='round,pad=0.4', facecolor=color, edgecolor=TEXT,
                alpha=0.85, linewidth=1.5)
    ax.text(x, y, text, ha='center', va='center', fontsize=11,
            color=BG, fontweight='bold', bbox=bbox)

# Level 0: individual characters
chars = ['t', 'h', 'e', ' ', 'c', 'a', 't']
xs = [0.5, 1.7, 2.9, 4.1, 5.3, 6.5, 7.7]
for x, ch in zip(xs, chars):
    token_box(ax, x, 0.5, repr(ch), color=MUTED)

# Level 1: after merging t+h → th
pairs1 = ['th', 'e', ' ', 'ca', 't']
xs1    = [1.1, 2.9, 4.1, 5.9, 7.7]
for x, tok in zip(xs1, pairs1):
    token_box(ax, x, 1.5, repr(tok), color=PURPLE)

# Level 2: after merging th+e → the
pairs2 = ['the', ' ', 'cat']
xs2    = [2.0, 4.1, 6.5]
for x, tok in zip(xs2, pairs2):
    token_box(ax, x, 2.5, repr(tok), color=BLUE)

# Level 3: after merging ' '+the → ' the'
token_box(ax, 3.1, 3.5, "' the'", color=ACCENT)
token_box(ax, 6.5, 3.5, "'cat'",  color=GREEN)

# Arrows
arrow_style = dict(arrowstyle='->', color=MUTED, alpha=0.5, lw=1.2)
for x_from, x_to, y_from, y_to in [
    (0.5, 1.1, 0.8, 1.2), (1.7, 1.1, 0.8, 1.2),
    (5.3, 5.9, 0.8, 1.2), (6.5, 5.9, 0.8, 1.2),
    (1.1, 2.0, 1.8, 2.2), (2.9, 2.0, 1.8, 2.2),
    (4.1, 3.1, 2.8, 3.2), (2.0, 3.1, 2.8, 3.2),
]:
    ax.annotate('', xy=(x_to, y_to), xytext=(x_from, y_from),
                arrowprops=arrow_style)

ax.text(5, 3.8, '← merge steps →', ha='center', color=MUTED, fontsize=9)
ax.set_title('BPE builds tokens bottom-up by merging frequent pairs',
             color=TEXT, fontsize=12)

plt.tight_layout()
plt.show()

## Part 3 — Encoding text with a learned vocabulary

Once BPE merges are learned, encoding new text is deterministic: apply the merges in order, greedily. The same sequence of characters always produces the same token sequence.

Modern tokenisers (GPT-2 uses a byte-level BPE variant) operate on **bytes** rather than characters, which gives a few important properties:
- Every possible input can be tokenised — there's no OOV problem at the byte level since any text is valid bytes
- Unicode characters that span multiple bytes get split at the byte level if they're rare enough
- The space before a word is typically included in the token — `" the"` and `"the"` are different tokens

That last point matters more than it seems.

In [ ]:
# ── Encode text using learned BPE merges ──────────────────────────────────────

def apply_bpe(text, merges):
    """
    Encode a text string using a list of BPE merge rules.
    Returns a list of token strings.
    """
    # Start with character-level split
    words = text.lower().split()
    encoded = []

    for word in words:
        chars = list(word) + ['</w>']
        # Apply each merge rule in order
        for pair, _ in merges:
            i = 0
            new_chars = []
            while i < len(chars):
                if (i < len(chars) - 1 and
                    chars[i] == pair[0] and chars[i+1] == pair[1]):
                    new_chars.append(''.join(pair))
                    i += 2
                else:
                    new_chars.append(chars[i])
                    i += 1
            chars = new_chars
        encoded.extend(chars)

    return encoded

# Build vocabulary from our trained merges
# Assign integer IDs to each unique token
all_tokens = set()
for word in vocab:
    all_tokens.update(word)
token_to_id = {tok: i for i, tok in enumerate(sorted(all_tokens))}
id_to_token = {i: tok for tok, i in token_to_id.items()}

# Test encoding
test_sentences = [
    "the cat sat on the mat",
    "a new runner runs low",
    "the flat mat",
]

print(f"Vocabulary size: {len(token_to_id)} tokens\n")

for sent in test_sentences:
    tokens = apply_bpe(sent, merges)
    ids    = [token_to_id.get(t, 0) for t in tokens]
    print(f"Input:   '{sent}'")
    print(f"Tokens:  {tokens}")
    print(f"IDs:     {ids}")
    print()

In [ ]:
# ── Visualise tokenisation as coloured spans ───────────────────────────────────
def visualise_tokens(text, tokens, ax, title):
    """Show which characters belong to which token."""
    colors = [BLUE, ACCENT, GREEN, YELLOW, PURPLE,
              '#ff9f43', '#00d2d3', '#ff6b6b', '#48dbfb']

    ax.set_xlim(0, len(tokens))
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.set_title(title, color=TEXT, fontsize=10)

    for i, tok in enumerate(tokens):
        display = tok.replace('</w>', '↵')
        color = colors[i % len(colors)]
        ax.add_patch(mpatches.FancyBboxPatch(
            (i + 0.05, 0.2), 0.9, 0.6,
            boxstyle='round,pad=0.05',
            facecolor=color, edgecolor=TEXT, alpha=0.85, linewidth=1
        ))
        ax.text(i + 0.5, 0.5, display, ha='center', va='center',
                fontsize=9, color=BG, fontweight='bold')
        ax.text(i + 0.5, 0.1, str(token_to_id.get(tok, '?')),
                ha='center', va='center', fontsize=7, color=MUTED)


sentences_to_show = [
    "the cat sat on the mat",
    "the runner runs lowest",
]

fig, axes = plt.subplots(len(sentences_to_show), 1,
                          figsize=(14, 2.5 * len(sentences_to_show)))

for ax, sent in zip(axes, sentences_to_show):
    toks = apply_bpe(sent, merges)
    visualise_tokens(sent, toks, ax,
                     f'"{sent}" → {len(toks)} tokens  (number below = token ID)')

fig.suptitle('Tokenisation: text → coloured token spans → integer IDs',
             color=TEXT, y=1.02)
plt.tight_layout()
plt.show()

## Part 4 — The vocabulary as a lookup table

After tokenisation, each token has an integer ID. This ID is the *only* thing the model sees — the string "cat" is invisible after this point. What the model receives is the number 1337 (or whatever ID was assigned to "cat"), which it will look up in an **embedding table** to get a vector.

That lookup table is a matrix of shape $(V, d)$ where $V$ is vocabulary size and $d$ is the embedding dimension. Row $i$ of the matrix is the learned vector representation of token $i$.

This is exactly what the Embeddings notebook (next in the series) covers. Tokenisation's job ends at the integer. Everything from here is the model's job.

```
"the cat"  →  ["the", " cat"]  →  [4, 182]  →  embedding lookup  →  vectors
  text          tokens            integer IDs       (next notebook)
```

The vocabulary size $V$ is a design choice that involves real tradeoffs:

| $V$ | Tokens per sentence | Embedding matrix size | Typical model |
|-----|---------------------|----------------------|---------------|
| 8,192 | Long | Small | Older models |
| 32,768 | Medium | Medium | GPT-2 (50k) |
| 100,000+ | Short | Large | GPT-4, Claude |

In [ ]:
# ── The vocabulary as a lookup table ──────────────────────────────────────────
V = len(token_to_id)
d = 8   # embedding dimension (tiny — real models use 768–12288)

# This is the embedding matrix — row i is the learned vector for token i
# In training, these are learned via backpropagation
# Here we initialise randomly just to show the structure
np.random.seed(0)
embedding_matrix = np.random.randn(V, d) * 0.1

sentence  = "the cat sat"
tokens    = apply_bpe(sentence, merges)
token_ids = [token_to_id.get(t, 0) for t in tokens]

print(f"Vocabulary size V = {V}")
print(f"Embedding dimension d = {d}")
print(f"Embedding matrix shape: ({V}, {d})\n")

print(f"Sentence: '{sentence}'")
print(f"Tokens:   {tokens}")
print(f"IDs:      {token_ids}\n")

print("Embedding lookup (each token ID → row of embedding matrix):")
for tok, tid in zip(tokens, token_ids):
    vec = embedding_matrix[tid]
    print(f"  '{tok}' (ID={tid}) → {vec.round(3)}")

print(f"\nFull sequence as matrix: shape = ({len(token_ids)}, {d})")
print("This matrix is the input to the transformer.")
print("→ Next notebook: what those vectors mean and how they're learned.")

## Part 5 — Non-obvious consequences of tokenisation

This is where understanding tokenisation pays off in practice. Several surprising LLM behaviours have direct tokenisation explanations.

### Arithmetic struggles

Numbers are tokenised in ways that don't reflect their mathematical structure. "1234" might be a single token, or it might be ["12", "34"], or ["1", "234"]. The model has no guarantee that "1234" and "1235" share any token, even though they differ by 1. Column-alignment in arithmetic is invisible at the token level.

### Spelling and character-level tasks

"How many 'r's are in strawberry?" is hard for a model not because counting is hard but because "strawberry" is a single token — the model never sees the individual characters unless the tokenisation happens to split it. The model has to recall character-level information from a token-level representation.

### Language inequality

Vocabularies trained on English-heavy corpora are optimised for English. A common English word like "the" is one token. An equivalent Chinese or Arabic word might require 2–4 tokens. This means non-English speakers pay more in context length (and cost) for the same semantic content.

### The leading space matters

" hello" (with leading space) and "hello" (without) are different tokens with different IDs. The model treats them differently — not because it understands capitalisation or word boundaries, but because they're literally different entries in the vocabulary table.

In [ ]:
# ── Demonstrate arithmetic tokenisation problem ────────────────────────────────

# Simulate how numbers get tokenised
# (Real GPT tokenisation: 1-3 digit numbers often get their own token,
# larger numbers get split in non-numeric ways)

numbers = ["5", "42", "137", "1234", "12345", "98765"]

# Simulate BPE tokenisation of numbers (simplified)
# The key point: tokenisation doesn't respect place value
def tokenise_number_sim(n_str):
    """Simulate how a BPE tokeniser might split number strings."""
    if len(n_str) <= 2:
        return [n_str]      # short numbers: one token
    elif len(n_str) == 3:
        return [n_str]      # 3-digit numbers: one token (common in vocabulary)
    elif len(n_str) == 4:
        return [n_str[:2], n_str[2:]]   # split at 2+2
    elif len(n_str) == 5:
        return [n_str[:3], n_str[3:]]   # split at 3+2
    else:
        return [n_str[:3], n_str[3:6], n_str[6:]]  # 3+3+rest

fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')
ax.set_title('Number tokenisation: place value is invisible to the model', color=TEXT)

colors = [BLUE, ACCENT, GREEN, YELLOW, PURPLE]
y_positions = np.linspace(0.85, 0.1, len(numbers))

for y, num in zip(y_positions, numbers):
    toks = tokenise_number_sim(num)
    ax.text(0.02, y, f'{num:>6s}  →', transform=ax.transAxes,
            color=TEXT, fontsize=11, va='center', fontfamily='monospace')
    x_offset = 0.22
    for i, tok in enumerate(toks):
        col = colors[i % len(colors)]
        bbox = dict(boxstyle='round,pad=0.3', facecolor=col, edgecolor=TEXT,
                    alpha=0.85, linewidth=1)
        ax.text(x_offset, y, tok, transform=ax.transAxes,
                color=BG, fontsize=11, fontweight='bold', va='center',
                bbox=bbox, fontfamily='monospace')
        x_offset += 0.06 + len(tok) * 0.018
    n_tok_label = f"← {len(toks)} token{'s' if len(toks)>1 else ''}"
    ax.text(x_offset + 0.01, y, n_tok_label, transform=ax.transAxes,
            color=MUTED, fontsize=9, va='center')

plt.tight_layout()
plt.show()

print("The model sees tokens, not digits.")
print("To add 1234 + 1, it must understand that token '12'+'34' represents")
print("a number whose last digit is '4' — a fact buried inside the token.")
print("This is why chain-of-thought prompting helps: it forces character-level work.")

In [ ]:
# ── Token efficiency across languages ─────────────────────────────────────────
# Simulate how a English-trained vocabulary handles different languages
# by measuring characters per token

# Representative sentences with approximate GPT-4 token counts
# (sourced from known tokenisation analyses)
language_examples = [
    ('English',  'The quick brown fox jumps over the lazy dog.',   9),
    ('French',   'Le renard brun rapide saute par-dessus le chien paresseux.', 13),
    ('Spanish',  'El rápido zorro marrón salta sobre el perro perezoso.', 12),
    ('German',   'Der schnelle braune Fuchs springt über den faulen Hund.', 12),
    ('Russian',  'Быстрая коричневая лиса прыгает через ленивую собаку.', 19),
    ('Arabic',   'الثعلب البني السريع يقفز فوق الكلب الكسول.', 22),
    ('Chinese',  '快速的棕色狐狸跳过了懒狗。', 14),
    ('Japanese', '素早い茶色のキツネが怠け者の犬を飛び越えた。', 20),
]

langs   = [l[0] for l in language_examples]
chars   = [len(l[1]) for l in language_examples]
tokens  = [l[2] for l in language_examples]
chars_per_tok = [c/t for c,t in zip(chars, tokens)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
bar_colors = [GREEN if l == 'English' else BLUE for l in langs]
ax.barh(langs, tokens, color=bar_colors, alpha=0.85)
ax.set_title('Tokens needed for equivalent sentence across languages\n(English-trained vocabulary)', color=TEXT)
ax.set_xlabel('Token count')
ax.axvline(tokens[0], color=GREEN, linestyle='--', alpha=0.5)
ax.grid(True, axis='x')

ax = axes[1]
ax.barh(langs, chars_per_tok, color=bar_colors, alpha=0.85)
ax.set_title('Characters per token — higher = more efficient tokenisation', color=TEXT)
ax.set_xlabel('Characters per token')
ax.axvline(chars_per_tok[0], color=GREEN, linestyle='--', alpha=0.5, label='English baseline')
ax.legend(fontsize=8)
ax.grid(True, axis='x')

plt.tight_layout()
plt.show()

print("English-trained vocabularies are highly optimised for English.")
print("Other languages require more tokens for equivalent content — meaning:")
print("  - Shorter effective context window")
print("  - Higher API cost per semantic unit")
print("  - Fewer tokens available for the model to 'think' in complex reasoning")

## Part 6 — Special tokens

Real tokenisers include **special tokens** that don't appear in training text but serve structural roles:

| Token | Typical usage |
|-------|---------------|
| `<\|endoftext\|>` | Marks the end of a document during training |
| `<\|pad\|>` | Padding to make sequences the same length in a batch |
| `[BOS]` / `[EOS]` | Beginning/end of sequence markers |
| `[SEP]` | Separator between segments (BERT-style) |
| `<\|im_start\|>` / `<\|im_end\|>` | Chat turn markers in instruction-tuned models |
| `<\|assistant\|>` | Role marker — tells the model it should generate as the assistant |

These special tokens are how the raw language model becomes a *chat* model. The instruction-tuned model learns during fine-tuning that text between `<|im_start|>user` and `<|im_end|>` is the human's input, and that it should generate text following `<|im_start|>assistant`.

When you send a message to an LLM API, your text doesn't arrive as raw text. It arrives wrapped in a **chat template** that inserts these structural tokens. The model's response ends when it generates the appropriate end-of-turn token.

In [ ]:
# ── Chat template: what the model actually sees ────────────────────────────────

# This is a simplified version of the ChatML format used by many models
def apply_chat_template(messages, system=None):
    """
    Convert a list of messages into the token sequence the model sees.
    This is what happens behind the scenes when you call a chat API.
    """
    result = []
    if system:
        result.append('<|im_start|>system')
        result.append(system)
        result.append('<|im_end|>')
    for msg in messages:
        result.append(f'<|im_start|>{msg["role"]}')
        result.append(msg['content'])
        result.append('<|im_end|>')
    result.append('<|im_start|>assistant')  # prompt model to respond
    return result


messages = [
    {'role': 'user',      'content': 'What is the capital of France?'},
    {'role': 'assistant', 'content': 'The capital of France is Paris.'},
    {'role': 'user',      'content': 'And what is its population?'},
]

formatted = apply_chat_template(
    messages,
    system='You are a helpful assistant.'
)

print("What the model actually receives (raw token sequence):")
print("─" * 60)
for part in formatted:
    if part.startswith('<|'):
        print(f'\033[33m{part}\033[0m')   # yellow for special tokens
    else:
        print(part)
print("─" * 60)
print("\nThe model generates text until it produces <|im_end|>")
print("That token is what stops generation — not a rule, a learned behaviour.")
print("\nConnection forward: the Sampling notebook covers what happens")
print("when the model generates token by token until this stop condition.")

## Part 7 — Token count vs context window

LLMs have a **context window** — a maximum number of tokens they can process in one forward pass. When people say "GPT-4 has a 128k context window" they mean 128,000 tokens, not 128,000 characters or words.

Because tokens are variable-length in characters:
- A context window of 128k tokens ≈ 100k–200k characters ≈ 50k–100k words (for English)
- For non-English languages with less efficient tokenisation, the same context window holds significantly less content
- Code is often tokenised more efficiently than prose because common patterns (`def`, `for`, `return`, `()`) have their own tokens

The context window is also why prompt engineering matters at the token level. If you can express an instruction in 10 tokens instead of 30, you've freed up 20 tokens for content — which matters when you're near the limit.

In [ ]:
# ── Visualise what fits in a context window ────────────────────────────────────
context_window = 128_000   # GPT-4 Turbo

content_types = [
    ('Short novel',         300_000, 'characters'),
    ('Legal contract',       50_000, 'characters'),
    ('News article',          3_000, 'characters'),
    ('Python script (1k lines)', 30_000, 'characters'),
    ('Academic paper',       60_000, 'characters'),
    ('Chat conversation\n(100 exchanges)', 20_000, 'characters'),
]

# Approximate characters per token by content type
chars_per_tok = {
    'Short novel': 4.0,
    'Legal contract': 4.2,
    'News article': 4.1,
    'Python script (1k lines)': 4.8,
    'Academic paper': 4.3,
    'Chat conversation\n(100 exchanges)': 3.8,
}

fig, ax = plt.subplots(figsize=(10, 5))

for i, (name, n_chars, unit) in enumerate(content_types):
    cpt = chars_per_tok[name]
    n_tokens = n_chars / cpt
    fits_frac = min(1.0, context_window / n_tokens)
    remainder = 1.0 - fits_frac

    ax.barh(i, fits_frac, color=GREEN if fits_frac >= 1 else BLUE, alpha=0.8)
    if remainder > 0:
        ax.barh(i, remainder, left=fits_frac, color=ACCENT, alpha=0.4)

    token_count = int(n_tokens)
    label = f'{token_count:,} tokens  ({"fits" if fits_frac >= 1 else f"{fits_frac:.0%} fits"})'
    ax.text(min(fits_frac, 1.0) + 0.01, i, label,
            va='center', color=TEXT, fontsize=9)

ax.set_yticks(range(len(content_types)))
ax.set_yticklabels([c[0] for c in content_types])
ax.axvline(1.0, color=YELLOW, linewidth=2, linestyle='--',
           label='Context window limit (128k tokens)')
ax.set_xlim(0, 1.8)
ax.set_title('What fits in a 128k token context window?', color=TEXT)
ax.set_xlabel('Fraction of content that fits')
ax.legend(fontsize=9)
ax.grid(True, axis='x')

plt.tight_layout()
plt.show()

## Where this leads

Tokenisation turns text into a sequence of integers. Those integers are the last moment anything about the original text is visible — everything downstream operates on numbers.

The next step is **Embeddings** (LLM Series — 2): those integers become dense vectors, and the geometry of those vectors encodes meaning. The embedding matrix is a learned parameter — which means the model is learning what "cat" means by learning where to place token 1337 in a high-dimensional space.

The connection to the rest of the series:
- The **Attention** notebook shows what happens to those embedding vectors as they flow through transformer blocks
- The **Markov Processes** notebook is relevant here: token generation is a Markov chain — the next token depends only on the current context, not the full history of how that context was built up
- The **RLHF** notebook explains how the model learns, after pretraining, to generate responses that end with the right stop tokens and follow the chat template correctly

---

## Summary

| Concept | What it is |
|---------|------------|
| **Tokenisation** | Converting text to integers — the entry point to every LLM |
| **BPE** | Learned subword vocabulary built by merging frequent character pairs |
| **Vocabulary size** | Design tradeoff between sequence length and embedding matrix size |
| **Subword units** | Rare words split into known pieces; common words get their own token |
| **Special tokens** | Structural markers: end-of-text, role boundaries, padding |
| **Chat template** | How raw messages become the structured token sequence the model sees |
| **Context window** | Maximum token count, not character count — language-dependent in practice |
| **Leading space** | `" hello"` and `"hello"` are different tokens — context affects tokenisation |

---

## Things to try

- Change `n_merges` to 5 vs 50 in the BPE training — how does the vocabulary change?
- Train BPE on code instead of prose — which merges appear first?
- Find a word that tokenises differently depending on whether it appears at the start of a sentence vs mid-sentence
- Estimate how many tokens your most recent long document would use at 4 chars/token
- Try encoding a number like 12345 character by character — what would it take for a model to correctly add 1 to it?